# Week 6: Probabilistic and Hierarchical Forecasting
## In-Class Exercises

**Objective.** Produce forecasts that carry honest uncertainty, check that the uncertainty is honest, and make a set of forecasts add up across a hierarchy.

### How this notebook works

Three parts, each building on the one before it.

| Part | Format | Content |
| --- | --- | --- |
| 1 | Walkthrough | Prediction intervals from a fitted model, and the coverage check that deflates them. |
| 2 | Blanks we fill in together | Three ways to build an 80% interval, scored against each other. |
| 3 | On your own, ~15 min | Quantile regression and pinball loss. |

In [ ]:
!pip install -q statsforecast lightgbm pandas matplotlib scikit-learn

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, SeasonalNaive

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
air = (pd.read_csv(URL, parse_dates=["Month"])
         .rename(columns={"Month": "ds", "Passengers": "y"})
         .assign(unique_id="airline"))

h, m = 24, 12
train, test = air.iloc[:-h], air.iloc[-h:]
train.tail(2)

---
## Part 1. An interval, and whether to believe it

A point forecast summarizes a distribution, and it is rarely the part of the distribution a decision needs. Inventory and capacity planning need a high quantile. A revenue forecast needs the mean *and* a downside.

`statsforecast` gives intervals with a `level` argument. Watch what happens when I check them.

In [ ]:
sf = StatsForecast(models=[AutoETS(season_length=m)], freq="MS")
fc = sf.forecast(df=train, h=h, level=[80, 95])
fc.head(3)

In [ ]:
ev = test.merge(fc, on=["unique_id", "ds"])

ax = train.set_index("ds")["y"].tail(48).plot(color="black", label="train")
ev.set_index("ds")["y"].plot(ax=ax, color="black", ls="--", label="actual")
ev.set_index("ds")["AutoETS"].plot(ax=ax, label="forecast")
ax.fill_between(ev["ds"], ev["AutoETS-lo-95"], ev["AutoETS-hi-95"], alpha=0.15, label="95%")
ax.fill_between(ev["ds"], ev["AutoETS-lo-80"], ev["AutoETS-hi-80"], alpha=0.3, label="80%")
ax.legend(fontsize=8)
plt.show()

for lv in [80, 95]:
    inside = ((ev["y"] >= ev[f"AutoETS-lo-{lv}"]) & (ev["y"] <= ev[f"AutoETS-hi-{lv}"])).mean()
    width = (ev[f"AutoETS-hi-{lv}"] - ev[f"AutoETS-lo-{lv}"]).mean()
    print(f"nominal {lv}%  ->  empirical coverage {inside:.0%}, mean width {width:.1f}")

**Notice:**

Coverage on 24 test points is a noisy estimate, so do not over-read one number. The direction of the miss is consistent across real datasets though: **published intervals are too narrow.** Three standard reasons:

1. The interval uses the model's estimate of $\sigma$ and ignores uncertainty in the *parameters*.
2. It ignores uncertainty about the *model*. You chose ETS after looking at the data; that choice had a variance too, and it appears nowhere in the arithmetic.
3. It assumes the future error distribution matches the past one. Structural breaks say otherwise.

The fan widening with horizon is right and important. A 12-month forecast with a constant-width band was built wrong.

---
## Part 2. Three ways to make an 80% interval

Same point forecast, three theories of uncertainty.

- **Normal**: $\hat{y}_{t+k} \pm z_{0.9} \cdot \hat\sigma_k$, with $\hat\sigma$ from in-sample residuals scaled by $\sqrt{k}$. Cheap, symmetric, Gaussian.
- **Bootstrap**: resample past residuals, roll them forward through the forecast path many times, take empirical quantiles. Keeps the residuals' shape and skew.
- **Split conformal**: hold out a calibration window, measure absolute errors *at each horizon*, use the 80th percentile as the half-width. Distribution-free, with a finite-sample guarantee under exchangeability.

Fill in the `TODO`s with me.

In [ ]:
# Point forecasts and in-sample residuals from the same model.
sf_fit = StatsForecast(models=[AutoETS(season_length=m)], freq="MS")
res = sf_fit.forecast(df=train, h=h, fitted=True)
fitted = sf_fit.forecast_fitted_values()

point = res["AutoETS"].to_numpy()
resid = (fitted["y"] - fitted["AutoETS"]).dropna().to_numpy()
actual = test["y"].to_numpy()
LEVEL = 80
alpha = 1 - LEVEL / 100

print("residual sd:", round(resid.std(), 2), " skew:", round(pd.Series(resid).skew(), 2))

In [ ]:
from scipy.stats import norm

# --- 1. Normal ---
z = norm.ppf(1 - alpha / 2)
sigma_k = resid.std() * np.sqrt(np.arange(1, h + 1))
normal_lo, normal_hi = point - z * sigma_k, point + z * sigma_k

# --- 2. Bootstrap ---
rng = np.random.default_rng(0)
B = 2000
# TODO - draw B paths. Each path is the point forecast plus a cumulative sum of
# residuals sampled with replacement (the cumsum is what makes the fan widen).
#   Hint: rng.choice(resid, size=(B, h), replace=True).cumsum(axis=1)
paths = ...
boot_lo = np.percentile(paths, 100 * alpha / 2, axis=0)
boot_hi = np.percentile(paths, 100 * (1 - alpha / 2), axis=0)

# --- 3. Split conformal ---
# Calibration: refit on data up to a cut point, forecast h steps, record |error| per horizon.
cal_errors = []
for cut in range(len(train) - 4 * h, len(train) - h, 3):
    sub = train.iloc[:cut]
    f = StatsForecast(models=[AutoETS(season_length=m)], freq="MS").forecast(df=sub, h=h)
    truth = train.iloc[cut:cut + h]["y"].to_numpy()
    cal_errors.append(np.abs(truth - f["AutoETS"].to_numpy()))
cal_errors = np.vstack(cal_errors)

# TODO - the conformal half-width at horizon k is the (1 - alpha) quantile of the
# absolute calibration errors at that horizon.
q_k = ...
conf_lo, conf_hi = point - q_k, point + q_k

<details>
<summary><b>Show the two filled-in lines</b></summary>

```python
paths = point + rng.choice(resid, size=(B, h), replace=True).cumsum(axis=1)

q_k = np.quantile(cal_errors, 1 - alpha, axis=0)
```
</details>

In [ ]:
def report(lo, hi, label):
    inside = ((actual >= lo) & (actual <= hi)).mean()
    return {"method": label, "coverage": f"{inside:.0%}", "mean width": round(np.mean(hi - lo), 1)}

print(pd.DataFrame([report(normal_lo, normal_hi, "normal"),
                    report(boot_lo, boot_hi, "bootstrap"),
                    report(conf_lo, conf_hi, "split conformal")]).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, (lo, hi, name) in zip(axes, [(normal_lo, normal_hi, "normal"),
                                     (boot_lo, boot_hi, "bootstrap"),
                                     (conf_lo, conf_hi, "conformal")]):
    ax.plot(test["ds"], actual, "k--", label="actual")
    ax.plot(test["ds"], point, label="point")
    ax.fill_between(test["ds"], lo, hi, alpha=0.25)
    ax.set_title(f"{name} {LEVEL}%")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

**Reading the table.**

- Coverage alone never settles the argument. An interval from minus infinity to plus infinity has perfect coverage and zero value. Report **coverage and width** as a pair.
- Bootstrap intervals inherit the residuals' asymmetry. If your errors skew, the interval should too, and the normal one cannot.
- Conformal's guarantee is real but conditional on **exchangeability**, which time series violate as soon as there is a trend or regime change. Calibration windows sitting just before the test period are the best available defense, not a proof.
- That calibration loop cost a dozen model fits. Uncertainty quantification is not free, which is the real reason it gets skipped.

---
## Part 3. Quantiles directly, and the loss that scores them

About 15 minutes.

Rather than deriving an interval from a mean forecast, model the quantiles directly. LightGBM does this with `objective="quantile", alpha=tau`.

The scoring rule is **pinball loss**. For quantile $\tau$, forecast $q$, actual $y$:

$$L_\tau(y, q) = \begin{cases}\tau\,(y - q) & y \ge q \\ (1 - \tau)\,(q - y) & y < q\end{cases}$$

It penalizes misses on the expensive side harder, and it is *minimized in expectation by the true $\tau$-quantile*. That property makes it the right scorer rather than an arbitrary one.

**Tasks.**

1. Finish `pinball(y, q, tau)`.
2. Build the lag feature matrix (lags 1, 12, 13) and fit LightGBM at $\tau \in \{0.1, 0.5, 0.9\}$.
3. Forecast the test window and plot all three quantile paths against the actuals.
4. Compute pinball loss at each $\tau$ for your model and for the AutoETS quantiles from Part 1, and say who wins where.
5. Check for crossing. If $q_{0.1} > q_{0.9}$ anywhere, explain the cause and the fix.

In [ ]:
from lightgbm import LGBMRegressor

def pinball(y, q, tau):
    """Mean pinball loss at quantile tau."""
    y, q = np.asarray(y, float), np.asarray(q, float)
    # TODO
    return ...


LAGS = [1, 12, 13]

def lag_matrix(series, lags=LAGS):
    s = pd.Series(np.asarray(series, float))
    X = pd.DataFrame({f"lag{L}": s.shift(L) for L in lags})
    keep = X.notna().all(axis=1)
    return X[keep], s[keep]

In [ ]:
# YOUR CODE HERE - tasks 2 through 5

<details>
<summary><b>Solution</b></summary>

```python
def pinball(y, q, tau):
    y, q = np.asarray(y, float), np.asarray(q, float)
    d = y - q
    return np.mean(np.maximum(tau * d, (tau - 1) * d))


ytr = train["y"].to_numpy()
X, tgt = lag_matrix(ytr)

taus = [0.1, 0.5, 0.9]
qfc = {}
for tau in taus:
    mdl = LGBMRegressor(objective="quantile", alpha=tau, n_estimators=300, verbosity=-1).fit(X, tgt)
    hist = list(ytr)
    preds = []
    for _ in range(h):
        row = [[hist[-L] for L in LAGS]]
        nxt = float(mdl.predict(row)[0])
        preds.append(nxt)
        hist.append(nxt)          # note: recursion uses the tau path, not the median path
    qfc[tau] = np.array(preds)

plt.plot(test["ds"], actual, "k--", label="actual")
for tau in taus:
    plt.plot(test["ds"], qfc[tau], label=f"tau={tau}")
plt.legend(); plt.show()

for tau in taus:
    ets_q = point + norm.ppf(tau) * sigma_k
    print(f"tau={tau}  LGBM {pinball(actual, qfc[tau], tau):.2f}   AutoETS {pinball(actual, ets_q, tau):.2f}")

crossed = (qfc[0.1] > qfc[0.9]).sum()
print("crossings:", crossed)
```

**On crossing.** The three models are fitted independently, so nothing enforces monotonicity. The cheap fix is sorting the quantiles at each timestamp (isotonic rearrangement), which is provably no worse under pinball loss. The principled fix is one model with a multi-quantile loss, which is what `MQLoss` did in Week 5.

**On recursion.** Feeding the $\tau = 0.9$ prediction back in compounds an *upper quantile* rather than a central path, so the 0.9 line drifts optimistically. Commonly missed. Recursive quantile forecasting needs simulation, not point recursion.
</details>

---
## Wrap-up

1. **Coverage and width, always together.** Either alone is gameable.
2. **Bootstrap keeps shape; conformal drops assumptions; normal is fast.** Pick knowingly.
3. **Pinball and CRPS are proper scoring rules.** They are how you compare distributions rather than points.
4. **Reconciliation is a projection.** MinT beats bottom-up because it uses information from every level.
5. **The quantile you report should come from the cost of being wrong**, not from convention.